# TARGE — Prismatic VLM (Colab)

End-to-end pipeline: setup → install → login → paths → data → overview → train → eval.


## 1. Setup: GPU check + mount Drive


In [ ]:
print("== GPU ==")
!nvidia-smi -L || echo "(no GPU detected)"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n== Drive ==")
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


## 2. Install dependencies


In [ ]:
import time, sys
%cd /content/drive/MyDrive/targe-prismatic-vlms
print("[install] pip install -e . (this can take ~1-2 min) ...")
t0 = time.time()
!pip install -e . --quiet
print(f"[install] done in {time.time()-t0:.1f}s")
# !pip install flash-attn --no-build-isolation --quiet  # optional speedup on A100/L4


## 3. Logins (Hugging Face, W&B)


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

# Hugging Face
hf_token = userdata.get('hf_token')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
who = whoami()
print(f"[hf] logged in as: {who.get('name')}  (orgs: {[o['name'] for o in who.get('orgs', [])]})")

# W&B (optional)
try:
    wandb_token = userdata.get('wandb_token')
    os.environ["WANDB_API_KEY"] = wandb_token
    print("[wandb] WANDB_API_KEY set from Colab secrets")
except Exception:
    print("[wandb] no wandb_token secret found — skipping (training will not log to W&B)")


## 4. Configure filepaths

All paths live here. Drive = persistent (slow); local = fast scratch on the Colab VM.


In [ ]:
import os
from pathlib import Path

# --- Drive (persistent) — write-only target for outputs ---
DRIVE_ROOT = Path("/content/drive/MyDrive/targe-prismatic-vlms")  # repo + outputs
DRIVE_RUNS = DRIVE_ROOT / "runs"                                  # checkpoints + train.log + eval results

# --- Local (Colab VM SSD) — data lives here, never copied back ---
TAR_PATH   = Path("/content/train_subset_5k.tar")                 # input tarball (already on VM)
DATA_DIR   = Path("/content/train_subset_5k")                     # untar destination (folder of same name)
CHAT_JSON  = DATA_DIR / "chat_subset_5k.json"

# HF cache: keep on VM, not Drive (avoids slow Drive writes during model download).
# Models redownload on each fresh VM, but that's ~30s for SmolLM2 and saves Drive I/O.
HF_CACHE = Path("/content/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"]      = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print(f"REPO (Drive) : {DRIVE_ROOT}")
print(f"RUNS (Drive) : {DRIVE_RUNS}    # checkpoints + metrics + logs go here")
print(f"TAR   (local): {TAR_PATH}      ({'exists' if TAR_PATH.exists() else 'MISSING'})")
print(f"DATA  (local): {DATA_DIR}      ({'extracted' if DATA_DIR.exists() else 'not yet extracted'})")
print(f"HF cache     : {HF_CACHE}      # local-only, not synced to Drive")


## 5. Untar dataset on the Colab VM

Source `/content/train_subset_5k.tar` is already on the VM (you uploaded it).
Extracts in place to `/content/train_subset_5k/` — no Drive read, no duplicate copy.


In [ ]:
import time, tarfile, subprocess

assert TAR_PATH.exists(), f"Tarball not found at {TAR_PATH} — upload it first."

if CHAT_JSON.exists():
    n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
    print(f"[skip] {DATA_DIR} already extracted ({n_files:,} files)")
else:
    print(f"[tar] extracting {TAR_PATH}  ({TAR_PATH.stat().st_size / 1e9:.2f} GB)")
    t0 = time.time()
    # Use the system tar — faster + shows progress with pv if available, fine without.
    !tar -xf "{TAR_PATH}" -C /content/
    dt = time.time() - t0
    n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
    print(f"[tar] done in {dt:.1f}s — {n_files:,} files at {DATA_DIR}")

# Quick layout sanity
print("\n== Top-level of DATA_DIR ==")
!ls -la "{DATA_DIR}" | head -20
assert CHAT_JSON.exists(), f"Expected {CHAT_JSON} after extraction — check tar contents."
print(f"\n[ok] chat json: {CHAT_JSON}  ({CHAT_JSON.stat().st_size / 1e6:.2f} MB)")


## 6. Dataset overview


In [ ]:
import json
from collections import Counter

n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
n_imgs  = sum(1 for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
n_dirs  = sum(1 for _ in DATA_DIR.iterdir() if _.is_dir())

with open(CHAT_JSON) as f:
    chat = json.load(f)

n_entries    = len(chat)
n_with_image = sum(1 for e in chat if e.get("image"))
n_text_only  = n_entries - n_with_image
turns        = Counter(len(e.get("conversations", [])) for e in chat)

print(f"Data dir          : {DATA_DIR}")
print(f"Top-level folders : {n_dirs}")
print(f"Total files       : {n_files:,}")
print(f"  image files     : {n_imgs:,}")
print(f"Chat JSON         : {CHAT_JSON.name}")
print(f"  total entries   : {n_entries:,}")
print(f"  with image      : {n_with_image:,}")
print(f"  text-only       : {n_text_only:,}")
print(f"  turns/example   : {dict(sorted(turns.items()))}")
print("\nSplit             : train-only (no held-out val in this subset)")


### Inspect a sample by image ID

Pass any substring of an image filename (e.g. `"0074375"`) and the cell renders the image + its chat turns.


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

_chat_cache = {}
def _load_chat():
    if "chat" not in _chat_cache:
        with open(CHAT_JSON) as f:
            _chat_cache["chat"] = json.load(f)
        _chat_cache["by_img"] = {e.get("image", ""): e for e in _chat_cache["chat"] if e.get("image")}
    return _chat_cache["chat"], _chat_cache["by_img"]

def show_sample(image_id: str, max_thumb=512):
    """`image_id` can be the full image path inside DATA_DIR, a filename, or any substring like '0074375'."""
    _, by_img = _load_chat()
    matches = [k for k in by_img if image_id in k]
    if not matches:
        print(f"[miss] no chat entry whose image path contains {image_id!r}")
        return
    if len(matches) > 1:
        print(f"[note] {len(matches)} matches — showing first. Others: {matches[1:6]}{'...' if len(matches)>6 else ''}")
    key = matches[0]
    entry = by_img[key]
    img_path = DATA_DIR / key

    display(Markdown(f"**id:** `{entry.get('id','?')}`  &nbsp;|&nbsp; **image:** `{key}`"))
    if img_path.is_file():
        im = Image.open(img_path)
        w, h = im.size
        if max(w, h) > max_thumb:
            im.thumbnail((max_thumb, max_thumb))
        display(im)
        display(Markdown(f"<sub>{w}×{h}px · {img_path.stat().st_size/1024:.1f} KB · `{img_path}`</sub>"))
    else:
        print(f"[warn] image not found on disk: {img_path}")

    md = ["**Conversation:**"]
    for turn in entry.get("conversations", []):
        role = turn.get("from", "?")
        text = turn.get("value", "").replace("<image>", "_<image>_")
        md.append(f"- **{role}:** {text}")
    display(Markdown("\n".join(md)))

# Example
show_sample("0074375")


## 7. Train (align stage)


In [ ]:
import os, time
os.environ["PYTHONUNBUFFERED"] = "1"

RUN_ID   = "targe-smollm2-5k"
LOG_PATH = DRIVE_RUNS / RUN_ID / "train.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
WB_KEY_VAL   = os.environ.get("WANDB_API_KEY", "").strip()
# Empty tokens no longer crash — pretrain.py purges empty HF_TOKEN env vars and falls back to anonymous.
# We still inline-inject them so distributed worker ranks definitely see them.

print(f"[train] run_id   : {RUN_ID}")
print(f"[train] log file : {LOG_PATH}")
print(f"[train] data     : /content/train_subset_5k/  (local, no Drive I/O)")
print(f"[train] wandb    : nbusich-northeastern-university/targe")
print(f"[train] hf_token : {'<set, ' + str(len(HF_TOKEN_VAL)) + ' chars>' if HF_TOKEN_VAL else '<empty — anonymous mode>'}")
print(f"[train] wb_key   : {'<set>' if WB_KEY_VAL else '<empty — wandb disabled>'}")
print(f"[train] started  : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

TRACKERS = "[jsonl,wandb]" if WB_KEY_VAL else "[jsonl]"

%cd /content/drive/MyDrive/targe-prismatic-vlms
!set -o pipefail; HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" WANDB_API_KEY="{WB_KEY_VAL}" HF_HOME="/content/hf_cache" \
  stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 scripts/pretrain.py \
    --model.type "targe-smollm2-360m-clipb-224px" \
    --stage align \
    --model.align_per_device_batch_size 14 \
    --model.align_global_batch_size 14 \
    --model.align_learning_rate 1e-4 \
    --model.align_max_steps 500 \
    --dataset.type "targe-subset-5k" \
    --run_id "{RUN_ID}" \
    --run_root_dir "{DRIVE_RUNS}" \
    --hf_token HF_TOKEN \
    --trackers '{TRACKERS}' \
    --wandb_entity nbusich-northeastern-university \
    --wandb_project targe 2>&1 | tee -a "{LOG_PATH}"

print(f"\n[train] finished : {time.strftime('%Y-%m-%d %H:%M:%S')}")


## 8. Eval (interactive generation against a checkpoint)


In [ ]:
RUN_ID   = "targe-smollm2-5k"
CKPT_DIR = DRIVE_RUNS / RUN_ID

ckpts = sorted((CKPT_DIR / "checkpoints").glob("*.pt"))
print(f"[eval] run dir   : {CKPT_DIR}")
print(f"[eval] checkpoints found ({len(ckpts)}):")
for c in ckpts:
    sz = c.stat().st_size / 1e9
    print(f"   - {c.name:40s}  {sz:6.2f} GB")

assert ckpts, f"No checkpoints found in {CKPT_DIR / 'checkpoints'}"
LATEST = ckpts[-1]
print(f"\n[eval] using latest: {LATEST.name}")


In [ ]:
%cd /content/drive/MyDrive/targe-prismatic-vlms
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="/content/hf_cache" \
  python scripts/generate.py --model_path "{CKPT_DIR}" --hf_token HF_TOKEN
